# PDD Mobility Scanner — Cross-Grade Estimation from Hike Photos

Uses [Depth Anything V2](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf) monocular depth estimation
to measure **cross-grade** (left-to-right slope) of the trail surface from photos captured by the scanner.

**Camera:** JX-F37 module (110° HFOV, 60° VFOV, f-theta lens, 1920×1080)

**Requirements:** GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

**Upload your hike `.jpg` images when prompted.**

## Step 1: Install dependencies

In [ ]:
!pip install -q transformers torch pillow matplotlib

## Step 2: Camera parameters

JX-F37 module specs. The lens follows an **f-theta** projection (r = f × θ),
so we derive the focal length from the horizontal FOV.

In [ ]:
import numpy as np

# JX-F37 camera module specs
IMG_W, IMG_H = 1920, 1080
HFOV_DEG = 110.0   # horizontal field of view
VFOV_DEG = 60.0    # vertical field of view
CX, CY = IMG_W / 2, IMG_H / 2  # principal point (image center)

# f-theta focal length in pixels:
# At the horizontal edge, r = 960 px and θ = 55°
# f = r / θ = 960 / radians(55) ≈ 1000 px
F_THETA = (IMG_W / 2) / np.radians(HFOV_DEG / 2)

print(f"f-theta focal length: {F_THETA:.1f} px")
print(f"FOV: {HFOV_DEG}° H x {VFOV_DEG}° V")

## Step 3: Upload hike photos

In [ ]:
from google.colab import files

uploaded = files.upload()
image_files = sorted(uploaded.keys())
print(f"Uploaded {len(image_files)} image(s): {image_files}")

## Step 4: Load model and run depth estimation

In [ ]:
from transformers import pipeline
from PIL import Image

pipe = pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")

results = []
for fname in image_files:
    image = Image.open(fname).convert("RGB")
    depth_output = pipe(image)
    depth_map = depth_output["depth"]
    depth_array = np.array(depth_map).astype(float)
    results.append({
        "filename": fname,
        "image": image,
        "depth_map": depth_map,
        "depth_array": depth_array,
    })
    print(f"{fname}: depth range [{depth_array.min():.0f}, {depth_array.max():.0f}], shape {depth_array.shape}")

print(f"\nProcessed {len(results)} image(s).")

## Step 5: Estimate cross-grade

For each image we:
1. Sample a horizontal band in the lower portion of the frame (where the trail surface is)
2. Average depth across the band to get a left-to-right depth profile
3. Use the f-theta camera model to project each pixel into 3D (X = across trail, Y = vertical)
4. Fit a line through the 3D cross-section — its slope is the cross-grade

Since depth is relative (not metric), both X and Y scale by the same factor,
so the slope ratio and angle are still valid.

In [ ]:
def estimate_cross_grade(depth_array, band_center_frac=0.80, band_height_frac=0.05,
                         trail_width_frac=0.60):
    """
    Estimate cross-grade from a depth map.

    Parameters
    ----------
    depth_array : 2D array of depth values
    band_center_frac : vertical position of the sampling band (0=top, 1=bottom).
                       0.80 looks at the trail ~2-3m ahead.
    band_height_frac : height of the band as a fraction of image height.
    trail_width_frac : fraction of the image width to use (centered).
                       Avoids edges where trail may not be visible.
    """
    h, w = depth_array.shape

    # --- 1. Extract horizontal band and average vertically ---
    band_center = int(h * band_center_frac)
    band_half = max(1, int(h * band_height_frac / 2))
    band_top = max(0, band_center - band_half)
    band_bot = min(h, band_center + band_half)
    depth_profile = np.mean(depth_array[band_top:band_bot, :], axis=0)  # (w,)

    # --- 2. Pixel coordinates for each column ---
    x = np.arange(w)
    dx = x - CX
    dy = band_center - CY  # scalar: vertical offset of the band

    # --- 3. f-theta projection to 3D ---
    r = np.sqrt(dx**2 + dy**2)           # distance from image center (px)
    theta = r / F_THETA                   # angle from optical axis (rad)

    # 3D ray direction (unit vector per column)
    safe_r = np.where(r > 0, r, 1)       # avoid division by zero at center
    ray_x = np.sin(theta) * dx / safe_r   # horizontal (across trail)
    ray_y = np.sin(theta) * dy / safe_r   # vertical

    # 3D positions (relative scale, but ratio is preserved)
    X = depth_profile * ray_x  # across trail
    Y = depth_profile * ray_y  # vertical

    # --- 4. Fit line to center portion of trail ---
    margin = (1 - trail_width_frac) / 2
    mask = (x >= w * margin) & (x < w * (1 - margin))

    coeffs = np.polyfit(X[mask], Y[mask], 1)
    slope_ratio = coeffs[0]
    cross_grade_deg = np.degrees(np.arctan(slope_ratio))
    cross_grade_pct = slope_ratio * 100

    return {
        "cross_grade_deg": cross_grade_deg,
        "cross_grade_pct": cross_grade_pct,
        "depth_profile": depth_profile,
        "X": X,
        "Y": Y,
        "band_top": band_top,
        "band_bot": band_bot,
        "fit_coeffs": coeffs,
        "mask": mask,
    }


# Run cross-grade estimation on each image
for r in results:
    cg = estimate_cross_grade(r["depth_array"])
    r["cross_grade"] = cg
    sign = "left-low" if cg["cross_grade_deg"] > 0 else "right-low"
    print(f"{r['filename']:30s}  cross-grade: {cg['cross_grade_deg']:+.2f}°  ({cg['cross_grade_pct']:+.1f}%)  [{sign}]")

## Step 6: Visualize

For each image: original photo, depth map with sampling band, and the cross-section profile with fitted line.

In [ ]:
import matplotlib.pyplot as plt

for r in results:
    cg = r["cross_grade"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- Original photo ---
    axes[0].imshow(r["image"])
    axes[0].axhline(cg["band_top"], color="cyan", linewidth=1, linestyle="--")
    axes[0].axhline(cg["band_bot"], color="cyan", linewidth=1, linestyle="--")
    axes[0].set_title(r["filename"])
    axes[0].axis("off")

    # --- Depth map with band ---
    im = axes[1].imshow(r["depth_array"], cmap="inferno")
    axes[1].axhline(cg["band_top"], color="cyan", linewidth=1, linestyle="--")
    axes[1].axhline(cg["band_bot"], color="cyan", linewidth=1, linestyle="--")
    axes[1].set_title("Depth map (sampling band in cyan)")
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    # --- Cross-section: 3D Y vs X with fit line ---
    X, Y, mask = cg["X"], cg["Y"], cg["mask"]
    axes[2].scatter(X[mask], Y[mask], s=1, alpha=0.5, label="trail cross-section")
    # fit line
    x_fit = np.array([X[mask].min(), X[mask].max()])
    y_fit = np.polyval(cg["fit_coeffs"], x_fit)
    axes[2].plot(x_fit, y_fit, "r-", linewidth=2,
                 label=f"cross-grade: {cg['cross_grade_deg']:+.2f}° ({cg['cross_grade_pct']:+.1f}%)")
    axes[2].set_xlabel("X (across trail, relative)")
    axes[2].set_ylabel("Y (vertical, relative)")
    axes[2].set_title("Trail cross-section")
    axes[2].legend()
    axes[2].set_aspect("equal")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Step 7: Summary statistics

In [ ]:
import pandas as pd

stats = []
for r in results:
    cg = r["cross_grade"]
    stats.append({
        "filename": r["filename"],
        "cross_grade_deg": round(cg["cross_grade_deg"], 2),
        "cross_grade_pct": round(cg["cross_grade_pct"], 1),
        "depth_min": r["depth_array"].min(),
        "depth_max": r["depth_array"].max(),
        "depth_mean": round(r["depth_array"].mean(), 1),
    })

df_stats = pd.DataFrame(stats)
df_stats

## Step 8: Export

In [ ]:
# Save depth maps
for r in results:
    out_name = r["filename"].rsplit(".", 1)[0] + "_depth.png"
    r["depth_map"].save(out_name)
    print(f"Saved {out_name}")

# Save stats with cross-grade
df_stats.to_csv("cross_grade_stats.csv", index=False)
print("Saved cross_grade_stats.csv")

# Download
files.download("cross_grade_stats.csv")
for r in results:
    out_name = r["filename"].rsplit(".", 1)[0] + "_depth.png"
    files.download(out_name)